# TissueSpectF on ColabA place to try the pipeline online, check a change, and look at the labels andthe baselines without touching a cluster.**What this notebook is for, and what it is not.** Colab gives you about twocores. The per-sample permutation stage (`maxt`) takes roughly an hour on 23cores for five cohorts, so here it would outlive the session — it is skippedthroughout, and the notebook says so wherever that matters. Several cellsrestrict the run to one or two chromosomes for speed. **Nothing produced underthose restrictions is a result.** It is a rehearsal: it proves the code runs andthe labels are right, and that is genuinely worth having before spending an houron the real thing.Runs top to bottom in about 15 minutes. Sections 1–3 need no data at all.

## 0. SetupUse **Runtime → Change runtime type → R** if it is offered. Otherwise this installs R.

In [ ]:
import shutil, subprocessif shutil.which("Rscript") is None:    print("Installing R (about two minutes)...")    subprocess.run("apt-get -qq update && apt-get -qq install -y r-base-core", shell=True)print(subprocess.run(["Rscript", "--version"], capture_output=True, text=True).stderr.strip())

In [ ]:
!git clone --depth 1 --quiet https://github.com/Danpc11/TissueSpectF.git 2>/dev/null || echo "already cloned"%cd TissueSpectF!chmod +x tsf!./tsf --help | head -30

## 1. Does the code work? (no data needed)The fastest way to check a change. 138 R tests plus the Python layer, then anend-to-end run on synthetic data with a known injected peak.

In [ ]:
!make test

In [ ]:
# Optional: the Python layer. numpy/pandas/scipy/scikit-learn only, no torch.!pip -q install pytest 2>/dev/null!python3 -m pytest tests/ml/ -q

### The self-checkBuilds two synthetic cohorts with components planted on purpose — one shared bythe whole tissue, one confined to a single condition — runs every stage, andasserts that the shared one comes back *exploratory* while the condition-specificone comes back *confirmed*. Roughly ten minutes.If a change breaks the pipeline, this is what tells you.

In [ ]:
!TSF_MAXT_B=100 TSF_CONDITION_B=300 ./tsf selfcheck 2>&1 | tail -30

## 2. The real cohorts: fetch and inspectFive liver series. `fetch` pulls what the configs declare; `check` reportsanything GEO named differently.

In [ ]:
import osos.environ["TSF_GEO_DIR"]     = "/content/data"os.environ["TSF_INTERIM_DIR"] = "/content/interim"os.environ["TSF_RESULTS_DIR"] = "/content/results"# To keep results after the session ends:# from google.colab import drive; drive.mount('/content/drive')# os.environ["TSF_RESULTS_DIR"] = "/content/drive/MyDrive/TissueSpectF/results"

In [ ]:
!./tsf fetch --geo-dir /content/data!./tsf check --geo-dir /content/data

### Read a series before trusting a configThis is the habit that has caught the most errors in this project: two groups apaper calls the same thing, a stage field that turns out to be a bin, anetiology column with a near-identical neighbour. Change the accession and thetwo fields to look at anything else.

In [ ]:
!Rscript scripts/inspect_series_matrix.R \    /content/data/GSE162694_series_matrix.txt.gz "fibrosis" "nas score"

## 3. Ingest and check the labelsThe one output to read carefully. Compare against the cohort table in theREADME: a count that differs means a label went somewhere unexpected, andnothing downstream is worth looking at until that is reconciled.

In [ ]:
!./tsf ingest 2>&1 | grep -E "Ingesting|Labels|: [0-9]+$|Grid|coverage|filter|keep_conditions"

In [ ]:
import pandas as pd, glob, osrows = []for f in sorted(glob.glob("/content/interim/*/samples.tsv")):    d = pd.read_csv(f, sep="\t")    d = d[d.get("keep", True).astype(bool)]    rows.append(d.groupby("class_id").size().rename(os.path.basename(os.path.dirname(f))))table = pd.concat(rows, axis=1).fillna(0).astype(int)table["total"] = table.sum(axis=1)table

## 4. Spectra, on part of the genome`--chromosomes` is what makes this feasible here. Every stage runs, on less ofthe genome. Change the list, or drop the flag if you have the patience.

In [ ]:
!./tsf run --from spectra --to spectra --chromosomes 1,17 2>&1 | tail -20

### Look at oneThe condition's summary spectrum for a chromosome: power against period ingenes. What to look at is whether anything stands above its neighbourhood, andat what scale.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt, globpaths = sorted(glob.glob("/content/results/*/spectra/spectra_condition_*.tsv"))print("\n".join(os.path.basename(p) for p in paths[:8]))d = pd.read_csv(paths[0], sep="\t")d = d[(d.chr.astype(str) == "17") & (d["sample"] == "avg_signal")]fig, ax = plt.subplots(figsize=(11, 3.5))ax.plot(d.period, d.power, lw=0.8)ax.set_xscale("log"); ax.set_xlabel("period (genes per cycle)")ax.set_ylabel("power"); ax.set_title(os.path.basename(paths[0]))ax.invert_xaxis(); plt.tight_layout(); plt.show()

### The spectral window: what the gaps alone can produceRun this before reading anything into a peak. A component sitting where thesampling pattern is strongest is an artefact until shown otherwise, and withcoverage well under 100% that is a live possibility rather than a formality.

In [ ]:
!./tsf window --chromosomes 1,17 2>&1 | tail -15

## 5. Consensus, and why the warnings matterPower, prevalence and phase-locking per frequency, from the per-sample spectrarather than from the spectrum of the mean — components present in every sampleat scattered phases cancel in the mean and vanish.`maxt` is skipped, so `prevalence_maxt` will be empty and the rank-basedstatistics carry the analysis. Permutations are set low for speed; the stageprints what that costs.

In [ ]:
!./tsf consensus --chromosomes 1,17 --n-null 199 2>&1 | tail -25

Two warnings are expected and are the pipeline telling you a claim is *notreachable*, which is different from absent:- *smallest reachable phase-alignment q* — a condition with too few samples to  test phase alignment at all. Expect it for the small classes.- *Pointwise null ... cannot confirm anything* — harmless. Confirmation uses the  family-wise `p_null_fwer`, whose floor is reachable.

## 6. The baselinesThis is the part worth running online even when nothing else is. It exports theper-sample spectra and runs a cohort-balanced nearest centroid, elastic net andrandom forest through leave-one-cohort-out — no torch, no model, minutes.Read `lift` over the training fold's majority class, not accuracy. And read theper-class report underneath: with these cohorts, some classes are evaluated bythree folds and others by none.

In [ ]:
!./tsf ae-prepare!python3 scripts/run_baselines.py --data /content/results/autoencoder/data \                                  --out  /content/results/autoencoder/baselines

**On a partial-genome run these numbers are a rehearsal.** They tell you theharness works and roughly what the fold structure looks like. The number thatdecides whether a learned model is worth building comes from the full run on thecluster — see `RUNBOOK.md`, section 9.

## 7. Try somethingA few things this notebook is a good place for:```bash# the other gene universe: does a component survive the definition change?./tsf run --from ingest --to spectra --gene-universe '^(protein-coding|ncRNA)$' \    --interim-dir /content/interim_nc --results-dir /content/results_nc \    --chromosomes 1,17# a different stability criterion, without recomputing anything upstream./tsf run --from stability --criterion consistency --stable-frac 0.7 --chromosomes 1,17# what a cohort contributes on its own./tsf ingest GSE130970```Every path and parameter is a flag; `./tsf --help` lists them. Precedence iscommand line over environment over `config/project.R`, and every override isechoed in the log.

## What this notebook cannot tell you- **Whether a component is real.** That needs the full genome, `maxt`, and the  permutation counts in `RUNBOOK.md`. A partial-genome rehearsal with 199 draws  is not evidence.- **Whether the classifier works.** Leave-one-cohort-out on two chromosomes is  not the same problem as on the whole grid.- **Whether a peak is biological.** Check `./tsf window` first, then whether it  replicates across cohorts with different coverage.What it does tell you: the code runs, the labels are what you think they are,and the machinery recovers a signal it is known to contain.